# Structured Output in LangChain

## Method 1 — `with_structured_output()`

`with_structured_output()` is the easiest and preferred way in LangChain to make an LLM return data in a predefined structure instead of free-form text.

---

# 1. Basic Idea

Normally:

    User Prompt
         ↓
        LLM
         ↓
    Free-form Text

With `with_structured_output()`:

    User Prompt
         ↓
        LLM
         ↓
    Structured Output
         ↓
    Python Object / Dictionary / JSON

The developer defines the required output schema, and LangChain uses that schema to guide the model's response.

---

# 2. Why Do We Need Structured Output?

LLMs normally return unstructured text.

Example:

    Prompt:
    "Give me information about a student named Ali."

Possible LLM response:

    Ali is 22 years old and is studying Computer Science.
    He lives in Mumbai and has a GPA of 8.5.

This is readable for humans but difficult for programs to process reliably.

We may instead want:

    {
        "name": "Ali",
        "age": 22,
        "course": "Computer Science",
        "city": "Mumbai",
        "gpa": 8.5
    }

Now the application can directly access individual fields:

    student["name"]
    student["age"]
    student["gpa"]

Therefore:

    Unstructured Text
          ↓
    Difficult for programs to process

    Structured Output
          ↓
    Easy for programs to process

---

# 3. Main Use Cases

Structured output is especially useful for:

1. Data Extraction
2. API Building
3. Agents
4. Database Operations
5. Classification
6. Information Extraction
7. Automation Pipelines

---

# 4. How `with_structured_output()` Works

Conceptually:

    Schema
       ↓
      LLM
       ↓
    User Prompt
       ↓
    Structured Response

The schema tells the model:

    "What fields should I return?"

Example:

    Student
    ├── name   : str
    ├── age    : int
    ├── city   : str
    └── gpa    : float

The LLM then tries to produce output matching this structure.

---

# 5. Example Using Pydantic

Pydantic can be used to define the output schema.

    from pydantic import BaseModel, Field
    from langchain_openai import ChatOpenAI

    class Student(BaseModel):
        name: str = Field(description="Student's name")
        age: int = Field(description="Student's age")
        course: str = Field(description="Student's course")
        city: str = Field(description="Student's city")

    model = ChatOpenAI(model="gpt-4o-mini")

    structured_model = model.with_structured_output(Student)

    result = structured_model.invoke(
        "Ali is 22 years old. He studies Computer Science and lives in Mumbai."
    )

    print(result)

Possible output:

    name='Ali'
    age=22
    course='Computer Science'
    city='Mumbai'

The important point is that the model is no longer simply returning arbitrary text.

It returns data following the `Student` schema.

---

# 6. Pydantic Schema

Pydantic is commonly used to define the structure.

Example:

    class Student(BaseModel):
        name: str
        age: int
        course: str

This means:

    Student
    ├── name   → string
    ├── age    → integer
    └── course → string

The schema defines:

- Field names
- Data types
- Optional descriptions
- Validation rules

---

# 7. Field Descriptions

We can provide descriptions using `Field()`.

    from pydantic import BaseModel, Field

    class Student(BaseModel):

        name: str = Field(
            description="Full name of the student"
        )

        age: int = Field(
            description="Age of the student"
        )

        course: str = Field(
            description="Course studied by the student"
        )

Descriptions help the model understand what each field represents.

---

# 8. Structured Output Flow

    ┌─────────────────────┐
    │       Schema        │
    │                     │
    │ name : str          │
    │ age  : int          │
    │ course : str        │
    └──────────┬──────────┘
               │
               ▼
         ┌─────────────┐
Prompt → │     LLM     │
         └──────┬──────┘
                │
                ▼
      ┌─────────────────────┐
      │ Structured Response │
      └──────────┬──────────┘
                 │
                 ▼
          Python Object
          / Dictionary
          / JSON

---

# 9. Normal Output vs Structured Output

## Normal LLM Output

    "Ali is 22 years old and studies Computer Science."

The developer must manually extract:

    name
    age
    course

This can become unreliable.

## Structured Output

    Student(
        name="Ali",
        age=22,
        course="Computer Science"
    )

The fields are already separated.

---

# 10. Advantages of Structured Output

## 10.1 Reliable Structure

The response follows a predefined schema.

## 10.2 Easy Parsing

You don't need to manually parse large text responses.

## 10.3 Type Safety

Pydantic can validate types.

For example:

    age: int

means the age should be an integer.

## 10.4 Easy Integration

Structured data can easily be passed to:

- APIs
- Databases
- Python functions
- Agents
- Data pipelines

## 10.5 Less Manual Parsing

Without structured output:

    LLM
      ↓
    Text
      ↓
    Regex / Parser
      ↓
    Data

With structured output:

    LLM
      ↓
    Structured Data

---

# 11. Dictionary Output

Depending on the model and structured-output method, the result can also be handled in a dictionary-like form.

Example:

    result = {
        "name": "Ali",
        "age": 22,
        "course": "Computer Science"
    }

Then:

    result["name"]

returns:

    Ali

---

# 12. Where Is Structured Output Used?

## 12.1 Data Extraction

Example: Extract information from a resume.

    Resume
       ↓
      LLM
       ↓
    Structured Schema
       ↓
    {
        name,
        email,
        skills,
        experience
    }

---

## 12.2 API Building

Suppose an API expects:

    {
        "product": "iPhone",
        "price": 79999,
        "rating": 4.5
    }

Structured output helps produce data in the expected format.

---

## 12.3 Agents

An agent may need to decide which tool should be used.

Example:

    User:
    "Calculate 25 × 10."

The agent may need structured information such as:

    {
        "tool": "calculator",
        "input": "25 * 10"
    }

Structured output makes the agent's decision more predictable.

---

## 12.4 Database Operations

Example:

    User Message
         ↓
        LLM
         ↓
    Structured Data
         ↓
      Database

Instead of storing a paragraph, we can store individual fields.

For example:

    {
        "name": "Ali",
        "age": 22,
        "city": "Mumbai"
    }

can be inserted into separate database columns.

---

# 13. Structured Output in Data Extraction

One of the most important real-world applications is extracting structured information from unstructured text.

Example:

    Input:

    "John is a 25-year-old software engineer
    working at ABC Technologies in Mumbai."

Schema:

    Person
    ├── name
    ├── age
    ├── profession
    ├── company
    └── city

Output:

    {
        "name": "John",
        "age": 25,
        "profession": "Software Engineer",
        "company": "ABC Technologies",
        "city": "Mumbai"
    }

This converts:

    Unstructured Text
           ↓
          LLM
           ↓
    Structured Information

---

# 14. Structured Output vs Normal LLM Output

| Feature | Normal Output | Structured Output |
|---|---|---|
| Format | Free-form text | Defined schema |
| Parsing | Often required | Usually minimal |
| Data extraction | Manual | Automatic/controlled |
| Validation | Manual | Can use Pydantic |
| API integration | Harder | Easier |
| Database insertion | Requires processing | Easier |
| Reliability | Lower | Higher |

---

# 15. Key Syntax

The most important syntax to remember:

    structured_model = model.with_structured_output(MySchema)

Then:

    result = structured_model.invoke("your prompt")

Basic pattern:

    Schema
       ↓
    model.with_structured_output(Schema)
       ↓
    invoke()
       ↓
    Structured Result

---

# 16. Important Example

    from pydantic import BaseModel
    from langchain_openai import ChatOpenAI

    class Product(BaseModel):
        name: str
        price: float
        rating: float

    model = ChatOpenAI(model="gpt-4o-mini")

    structured_model = model.with_structured_output(Product)

    result = structured_model.invoke(
        "The iPhone costs 79999 rupees and has a rating of 4.5."
    )

    print(result)

Possible output:

    Product(
        name="iPhone",
        price=79999.0,
        rating=4.5
    )

The LLM converts natural language into structured data.

---

# 17. Important Concept

`with_structured_output()` does NOT mean that the LLM itself becomes a database.

The LLM still generates the response.

LangChain provides the mechanism for requesting and handling the response according to the specified schema.

Think of it as:

    LLM
      +
    Schema
      +
    LangChain Structured Output
      ↓
    Structured Response

---

# 18. Structured Output vs Output Parser

There are two common approaches.

## Approach 1 — `with_structured_output()`

    model
      ↓
    with_structured_output(Schema)
      ↓
    structured result

This is the simpler and commonly preferred approach when the model supports structured output.

## Approach 2 — Output Parser

Conceptually:

    Prompt
      ↓
    LLM
      ↓
    Output Parser
      ↓
    Structured Data

An output parser processes the model's response and converts it into the required format.

---

# 19. Why `with_structured_output()` Is Useful in LangChain

LangChain applications often connect multiple components:

    User
      ↓
    Prompt
      ↓
    LLM
      ↓
    Structured Output
      ↓
    Python Logic
      ↓
    Tool / API / Database
      ↓
    Final Result

If the LLM output is unpredictable, the next component may fail.

Structured output creates a predictable interface between the LLM and the rest of the application.

---

# 20. Structured Output as an Interface

Think of the schema as a contract.

    ┌───────────────┐
    │     LLM       │
    └───────┬───────┘
            │
            │ must follow
            ▼
    ┌───────────────┐
    │    Schema     │
    │               │
    │ name : str    │
    │ age  : int    │
    │ city : str    │
    └───────┬───────┘
            │
            ▼
    Application Code

The schema tells the application what kind of data it expects.

---

# 21. Structured Output and Validation

Structured output becomes especially powerful when combined with Pydantic validation.

Example:

    class Student(BaseModel):
        name: str
        age: int

If the application expects:

    age → integer

but receives invalid data, validation can identify the problem.

Therefore:

    LLM Output
       ↓
    Schema
       ↓
    Validation
       ↓
    Valid Structured Data

This is important for production applications.

---

# 22. Real-World Example — Resume Parser

Suppose a user uploads:

    "Musharraf is a Python developer with
    2 years of experience and skills in
    Python, SQL and Power BI."

We define:

    class Resume(BaseModel):
        name: str
        role: str
        experience: int
        skills: list[str]

The model can produce:

    {
        "name": "Musharraf",
        "role": "Python Developer",
        "experience": 2,
        "skills": [
            "Python",
            "SQL",
            "Power BI"
        ]
    }

Now this information can be:

    Structured Data
          ↓
    Database
          ↓
    Search
          ↓
    Analytics
          ↓
    Recommendation System

---

# 23. Real-World Example — Sentiment Analysis

Input:

    "The product is excellent and I really like it."

Schema:

    class Sentiment(BaseModel):
        sentiment: str
        score: float

Output:

    {
        "sentiment": "positive",
        "score": 0.95
    }

This can then be stored in a database or used by another application.

---

# 24. Real-World Example — Product Extraction

Input:

    "The laptop has 16GB RAM, an Intel i7 processor
    and costs ₹75,000."

Schema:

    Product
    ├── name
    ├── ram
    ├── processor
    └── price

Output:

    {
        "name": "Laptop",
        "ram": "16GB",
        "processor": "Intel i7",
        "price": 75000
    }

---

# 25. Structured Output in Agents

Agents frequently need structured information because they interact with tools.

Example:

    User
      ↓
    Agent
      ↓
    Decide Action
      ↓
    Structured Output
      ↓
    Tool
      ↓
    Result

Example decision:

    {
        "tool": "calculator",
        "arguments": {
            "expression": "25 * 10"
        }
    }

This makes tool calling and agent workflows more reliable.

---

# 26. Structured Output in APIs

Imagine an API endpoint expects:

    {
        "name": "Ali",
        "age": 22,
        "city": "Mumbai"
    }

The LLM can extract these values from:

    "My name is Ali. I am 22 years old
    and I live in Mumbai."

Flow:

    Natural Language
          ↓
          LLM
          ↓
    Structured Output
          ↓
          API
          ↓
    Application

---

# 27. Structured Output in Database Systems

Example:

    User:
    "Add John, age 25, from Mumbai."

LLM:

    {
        "name": "John",
        "age": 25,
        "city": "Mumbai"
    }

Application:

    INSERT INTO users
    (name, age, city)
    VALUES
    ("John", 25, "Mumbai");

Therefore:

    Natural Language
          ↓
         LLM
          ↓
    Structured Data
          ↓
       Database

---

# 28. Structured Output in Automation

Structured output is useful when the output becomes the input of another system.

Example:

    Email
      ↓
    LLM
      ↓
    {
        "category": "complaint",
        "priority": "high",
        "customer": "Ali"
    }
      ↓
    Ticketing System

This is much easier than trying to interpret free-form text programmatically.

---

# 29. Important Terms

### Schema

Defines the expected structure of the output.

Example:

    Student
    ├── name
    ├── age
    └── course

### Field

An individual property inside the schema.

Example:

    name: str

### Type

Defines what kind of value a field should contain.

Examples:

    str
    int
    float
    bool
    list

### Validation

Checks whether the output follows the expected rules.

### Structured Response

The final response that follows the predefined schema.

---

# 30. Mental Model

Remember this simple diagram:

    ┌────────────────────┐
    │   Natural Language │
    │       Prompt       │
    └─────────┬──────────┘
              ↓
    ┌────────────────────┐
    │        LLM         │
    └─────────┬──────────┘
              ↓
    ┌────────────────────┐
    │      Schema        │
    │                    │
    │ name : str         │
    │ age  : int         │
    │ city : str         │
    └─────────┬──────────┘
              ↓
    ┌────────────────────┐
    │ Structured Output  │
    └─────────┬──────────┘
              ↓
       Application Logic
              ↓
       API / Database /
       Agent / Tool

---

# 31. Quick Revision

## What is Structured Output?

Structured output means making an LLM return information in a predefined format instead of arbitrary text.

## Why?

Because applications need predictable and machine-readable data.

## Main LangChain Method

    model.with_structured_output(Schema)

## Common Schema Tool

    Pydantic

## Basic Flow

    Prompt
      ↓
    LLM
      ↓
    Schema
      ↓
    Structured Response
      ↓
    Application

## Main Benefits

- Predictable output
- Easier parsing
- Validation
- Type safety
- Easier API integration
- Easier database integration
- Useful for agents
- Useful for automation
- Useful for data extraction

---

# 32. One-Line Definition

> `with_structured_output()` is a LangChain method that allows an LLM to return responses according to a predefined schema, making the output easier to validate, parse, and use programmatically.

---

# 33. Most Important Code to Remember

    from pydantic import BaseModel
    from langchain_openai import ChatOpenAI

    class Student(BaseModel):
        name: str
        age: int
        course: str

    model = ChatOpenAI(model="gpt-4o-mini")

    structured_model = model.with_structured_output(Student)

    result = structured_model.invoke(
        "Ali is 22 years old and studies Computer Science."
    )

    print(result)

Possible output:

    Student(
        name="Ali",
        age=22,
        course="Computer Science"
    )

---

# 34. Final Mental Shortcut

    Normal LLM:

    Prompt → LLM → Text


    Structured Output:

    Prompt
       ↓
      LLM
       ↓
     Schema
       ↓
    Structured Data


Remember:

    `with_structured_output()`
            =
    "Tell the LLM what structure the output should follow."


# Annotated TypedDict

## 1. What is TypedDict?

`TypedDict` is a feature from Python's `typing` module that allows us to define the expected **structure of a dictionary**.

A normal Python dictionary does not tell Python or a developer:

- Which keys should exist.
- What type of value each key should contain.
- Which fields are expected.

Example of a normal dictionary:

    person = {
        "name": "Musharraf",
        "age": 25
    }

With `TypedDict`, we can describe the expected structure:

    from typing import TypedDict

    class Person(TypedDict):
        name: str
        age: int

Now the intended structure is:

    Person
    ├── name → str
    └── age  → int

---

# 2. What is Annotated?

`Annotated` is provided by Python's `typing` module.

It allows us to attach **additional metadata** to a type.

Basic syntax:

    Annotated[type, metadata]

Example:

    from typing import Annotated

    name: Annotated[str, "Person's name"]

Here:

    str
    ↓
    Actual type

    "Person's name"
    ↓
    Metadata / additional information

So:

    Annotated[str, "Person's name"]

means:

> The value is a `str`, and `"Person's name"` is additional metadata associated with that type.

---

# 3. What is Annotated TypedDict?

`Annotated TypedDict` means using `Annotated` inside a `TypedDict`.

It allows us to define:

1. The dictionary structure.
2. The type of each field.
3. Additional metadata about each field.

Example:

    from typing import TypedDict, Annotated

    class Person(TypedDict):
        name: Annotated[str, "The person's name"]
        age: Annotated[int, "The person's age"]

This tells us:

    Person
    ├── name
    │   ├── type → str
    │   └── metadata → "The person's name"
    │
    └── age
        ├── type → int
        └── metadata → "The person's age"

---

# 4. Basic Syntax

General syntax:

    class ClassName(TypedDict):
        field_name: Annotated[type, metadata]

Example:

    from typing import TypedDict, Annotated

    class Student(TypedDict):
        name: Annotated[str, "Student's name"]
        age: Annotated[int, "Student's age"]
        grade: Annotated[str, "Student's grade"]

Expected structure:

    {
        "name": "Ali",
        "age": 21,
        "grade": "A"
    }

---

# 5. Why Use Annotated TypedDict?

`Annotated TypedDict` is useful when you want more information than just the basic type.

A normal `TypedDict` gives:

    name → str
    age → int

An `Annotated TypedDict` can give:

    name → str + description
    age → int + description

For example:

    class Product(TypedDict):
        name: Annotated[str, "Name of the product"]
        price: Annotated[float, "Price of the product"]
        quantity: Annotated[int, "Number of items available"]

Now every field has:

- A type.
- Additional metadata.

---

# 6. TypedDict vs Annotated TypedDict

## Normal TypedDict

    from typing import TypedDict

    class Person(TypedDict):
        name: str
        age: int

The structure is:

    name → str
    age  → int

## Annotated TypedDict

    from typing import TypedDict, Annotated

    class Person(TypedDict):
        name: Annotated[str, "Person's name"]
        age: Annotated[int, "Person's age"]

The structure is:

    name → str + metadata
    age  → int + metadata

### Main Difference

    TypedDict
        ↓
    Structure + type information

    Annotated TypedDict
        ↓
    Structure + type information + metadata

---

# 7. What is Metadata?

Metadata means **additional information attached to a type**.

Example:

    age: Annotated[int, "Age of the person"]

Here:

    int
    ↓
    Type

    "Age of the person"
    ↓
    Metadata

Metadata can be useful to tools, libraries, frameworks, documentation systems, or application code that knows how to interpret it.

Important:

> Metadata is not automatically a validation rule.

---

# 8. Important: Annotated Does NOT Automatically Validate Data

This is one of the most important points to remember.

Consider:

    class Person(TypedDict):
        age: Annotated[int, "Age must be positive"]

It may look like `"Age must be positive"` is a validation rule.

It is NOT automatically a validation rule.

`Annotated` simply attaches metadata.

It does not automatically enforce:

    age > 0

For example, Python does not automatically reject:

    person = {
        "age": -10
    }

because of the metadata:

    "Age must be positive"

If you need actual runtime validation, use a validation system such as **Pydantic**.

---

# 9. Annotated TypedDict in LLM Applications

`Annotated TypedDict` becomes particularly useful when working with **LLMs and structured output**.

An LLM normally produces free-form text.

Example:

    "The movie is Inception. It was released in 2010
    and has a rating of 8.8."

For an application, we may want structured data:

    {
        "title": "Inception",
        "year": 2010,
        "rating": 8.8
    }

We can define the expected structure using `TypedDict`.

Example:

    from typing import TypedDict

    class Movie(TypedDict):
        title: str
        year: int
        rating: float

We can also add metadata:

    from typing import TypedDict, Annotated

    class Movie(TypedDict):
        title: Annotated[str, "Movie title"]
        year: Annotated[int, "Movie release year"]
        rating: Annotated[float, "Movie rating"]

---

# 10. Why Metadata Can Be Useful for LLMs

When building structured-output systems, descriptions of fields can provide additional information about what each field represents.

Example:

    class Movie(TypedDict):
        title: Annotated[str, "Name of the movie"]
        year: Annotated[int, "Year in which the movie was released"]
        rating: Annotated[float, "Rating of the movie from 0 to 10"]

The structure now communicates:

    title
    → string
    → movie name

    year
    → integer
    → release year

    rating
    → float
    → movie rating

This makes the intended schema more descriptive.

---

# 11. Example: Sentiment Analysis

Suppose an LLM analyzes customer feedback.

We want:

    {
        "sentiment": "positive",
        "reason": "The customer liked the product."
    }

We can define:

    from typing import TypedDict, Annotated

    class Sentiment(TypedDict):
        sentiment: Annotated[str, "Overall sentiment of the text"]
        reason: Annotated[str, "Reason for the sentiment"]

Expected structure:

    {
        "sentiment": "positive",
        "reason": "The customer liked the product."
    }

Important:

`Annotated` describes the fields, but by itself it does NOT enforce that:

    sentiment ∈ {"positive", "neutral", "negative"}

For strict validation, another mechanism is required.

---

# 12. Annotated + Literal

`Annotated` can also be combined with `Literal`.

`Literal` restricts a value to a predefined set of values.

Example:

    from typing import TypedDict, Annotated, Literal

    class Sentiment(TypedDict):
        sentiment: Annotated[
            Literal["positive", "neutral", "negative"],
            "Overall sentiment"
        ]

Now the intended values are:

    positive
    neutral
    negative

The important distinction is:

    Literal
        ↓
    Restricts the allowed type/value options

    Annotated
        ↓
    Adds metadata

Together:

    Annotated[
        Literal["positive", "neutral", "negative"],
        "Overall sentiment"
    ]

---

# 13. Multiple Metadata Values

`Annotated` can contain more than one metadata item.

Example:

    from typing import Annotated

    age: Annotated[
        int,
        "Person's age",
        "Must be a whole number"
    ]

The first argument is the actual type:

    int

The remaining arguments are metadata:

    "Person's age"
    "Must be a whole number"

General form:

    Annotated[type, metadata1, metadata2, ...]

---

# 14. Nested Structures

`TypedDict` can contain other structured types.

Example:

    from typing import TypedDict, Annotated

    class Address(TypedDict):
        city: Annotated[str, "City name"]
        country: Annotated[str, "Country name"]

    class Person(TypedDict):
        name: Annotated[str, "Person's name"]
        address: Address

Expected structure:

    {
        "name": "Ali",
        "address": {
            "city": "Mumbai",
            "country": "India"
        }
    }

This becomes useful when the structured output contains nested objects.

---

# 15. Lists with Annotated

You can also use `Annotated` with collection types.

Example:

    from typing import TypedDict, Annotated

    class Student(TypedDict):
        name: Annotated[str, "Student name"]
        subjects: Annotated[list[str], "List of subjects"]

Expected structure:

    {
        "name": "Ali",
        "subjects": [
            "Python",
            "SQL",
            "Machine Learning"
        ]
    }

---

# 16. Optional Fields

`TypedDict` can also contain optional values using `NotRequired`.

Example:

    from typing import TypedDict, Annotated, NotRequired

    class Person(TypedDict):
        name: Annotated[str, "Person's name"]
        age: Annotated[int, "Person's age"]
        email: NotRequired[Annotated[str, "Email address"]]

Here:

    name → required
    age → required
    email → optional

The `email` key does not have to be present.

---

# 17. Required vs Optional Fields

By default, fields in a `TypedDict` are required.

Example:

    class Person(TypedDict):
        name: str
        age: int

Both fields are expected:

    {
        "name": "Ali",
        "age": 25
    }

For an optional key:

    from typing import TypedDict, NotRequired

    class Person(TypedDict):
        name: str
        age: int
        email: NotRequired[str]

Now this is also valid:

    {
        "name": "Ali",
        "age": 25
    }

because `email` is not required.

---

# 18. TypedDict Does Not Create a Runtime Class Like Pydantic

This is an important conceptual difference.

`TypedDict` mainly describes the expected structure for:

- Type checkers.
- Developers.
- IDEs.
- Static analysis.

It does not turn the dictionary into a special runtime object in the same way a Pydantic model does.

Example:

    class Person(TypedDict):
        name: str
        age: int

    person = {
        "name": "Ali",
        "age": 25
    }

The actual data is still a normal Python dictionary.

---

# 19. TypedDict and Runtime Validation

Remember:

    TypedDict
        ↓
    Type information / expected structure
        ↓
    Not runtime validation

Example:

    class Person(TypedDict):
        age: int

This does not mean Python automatically checks:

    age == int

at runtime whenever a dictionary is created.

For strong runtime validation:

    Pydantic
        ↓
    Runtime validation + parsing

---

# 20. Annotated TypedDict vs Pydantic

## Annotated TypedDict

Use when:

- You mainly need structure.
- You need type hints.
- You want metadata.
- You want a lightweight Python typing solution.
- You do not need complex runtime validation.

Example:

    class Person(TypedDict):
        name: Annotated[str, "Person's name"]
        age: Annotated[int, "Person's age"]

## Pydantic

Use when:

- You need runtime validation.
- You need default values.
- You need constraints.
- You need automatic type conversion.
- You need parsing.
- You need a proper validated Python object.

Example:

    from pydantic import BaseModel

    class Person(BaseModel):
        name: str
        age: int

---

# 21. TypedDict vs Pydantic vs JSON Schema

| Feature | TypedDict | Annotated TypedDict | Pydantic | JSON Schema |
|---|---|---|---|---|
| Basic structure | ✅ | ✅ | ✅ | ✅ |
| Type information | ✅ | ✅ | ✅ | ✅ |
| Field metadata | ❌ | ✅ | ✅ | ✅ |
| Runtime validation | ❌ | ❌ by itself | ✅ | ✅ |
| Automatic conversion | ❌ | ❌ | ✅ | Depends on validator |
| Default values | Limited/typing-based | Limited/typing-based | ✅ | Schema-dependent |
| Python object | ❌ | ❌ | ✅ | ❌ |
| Python-native | ✅ | ✅ | ✅ | ❌ |
| Lightweight | ✅ | ✅ | Moderate | ✅ |
| Useful for LLM schemas | ✅ | ✅ | ✅ | ✅ |

---

# 22. When to Use TypedDict

Use `TypedDict` when:

- You mainly need a dictionary structure.
- You want static type hints.
- You don't need runtime validation.
- You trust the incoming data.
- You want a lightweight solution.

Example:

    class User(TypedDict):
        name: str
        age: int

---

# 23. When to Use Annotated TypedDict

Use `Annotated TypedDict` when:

- You need a dictionary structure.
- You want type information.
- You want to attach metadata to fields.
- Field descriptions are useful.
- You are working with frameworks that understand `Annotated`.
- You don't necessarily need runtime validation.

Example:

    class User(TypedDict):
        name: Annotated[str, "User's full name"]
        age: Annotated[int, "User's age"]

---

# 24. When to Use Pydantic

Use Pydantic when:

- Data must be validated.
- You need constraints.
- You need default values.
- You need automatic type conversion.
- You need custom validation.
- You want structured Python objects.
- Reliability of incoming data is important.

Example:

    from pydantic import BaseModel, Field

    class User(BaseModel):
        name: str
        age: int = Field(gt=0)

Here Pydantic can actually enforce the constraint:

    age > 0

This is different from simply writing:

    age: Annotated[int, "Age must be positive"]

because the latter is metadata and does not itself enforce the condition.

---

# 25. When to Use JSON Schema

Use JSON Schema when:

- You want a standard JSON-based schema.
- You need schema-based validation.
- You don't want to depend specifically on Pydantic.
- You need a schema that can be used outside Python.
- You want interoperability between different languages/tools.

JSON Schema is especially useful when the schema itself needs to be represented as JSON.

---

# 26. Structured Output Hierarchy

When working with LLMs, think about structured output like this:

    LLM
     ↓
    Need structured response
     ↓
    Define schema
     ↓
    ┌───────────────────────────────┐
    │                               │
    │ TypedDict                     │
    │ → Basic structure              │
    │                               │
    │ Annotated TypedDict            │
    │ → Structure + metadata         │
    │                               │
    │ Pydantic                      │
    │ → Structure + validation       │
    │                               │
    │ JSON Schema                   │
    │ → Standard JSON schema         │
    │                               │
    └───────────────────────────────┘

---

# 27. Important Mental Model

Think of these concepts as different levels:

    TypedDict
        ↓
    "What fields should exist?"

    Annotated
        ↓
    "What additional information describes this field?"

    Pydantic
        ↓
    "Is the actual data valid?"

    JSON Schema
        ↓
    "What is the standard JSON representation
     of the expected structure?"

---

# 28. Common Mistake

### Mistake:

Thinking this:

    age: Annotated[int, "Age must be positive"]

automatically validates that the age is positive.

### Reality:

    int
    ↓
    Type information

    "Age must be positive"
    ↓
    Metadata

There is no automatic:

    age > 0

validation just because the description says so.

Use Pydantic or another validation mechanism for actual validation.

---

# 29. Another Common Mistake

### Mistake:

Thinking `TypedDict` converts a dictionary into a special object.

### Reality:

The actual value remains a dictionary.

`TypedDict` mainly provides typing information.

---

# 30. Key Advantages

## Advantages of Annotated TypedDict

- Simple.
- Lightweight.
- Python-native.
- Good type hints.
- Clear dictionary structure.
- Allows field metadata.
- Useful for structured data definitions.
- Useful in LLM applications.
- Works well with static type checking.
- Can be combined with `Literal`, `NotRequired`, lists, nested types, etc.

---

# 31. Limitations

## Limitations of Annotated TypedDict

- No automatic runtime validation.
- Metadata does not automatically enforce constraints.
- No automatic type conversion.
- Does not provide the rich validation functionality of Pydantic.
- Its usefulness depends on whether the framework/tool consuming the metadata understands it.

---

# 32. Complete Example

    from typing import TypedDict, Annotated, Literal, NotRequired

    class Movie(TypedDict):
        title: Annotated[str, "Name of the movie"]
        year: Annotated[int, "Release year"]
        rating: Annotated[float, "Movie rating"]
        sentiment: Annotated[
            Literal["positive", "neutral", "negative"],
            "Overall movie sentiment"
        ]
        review: NotRequired[
            Annotated[str, "Optional movie review"]
        ]

Expected structure:

    {
        "title": "Inception",
        "year": 2010,
        "rating": 8.8,
        "sentiment": "positive",
        "review": "Excellent movie."
    }

Structure:

    Movie
    ├── title
    │   └── str + metadata
    │
    ├── year
    │   └── int + metadata
    │
    ├── rating
    │   └── float + metadata
    │
    ├── sentiment
    │   └── Literal + metadata
    │
    └── review
        └── Optional + Annotated str

---

# 33. Revision Table

| Concept | Meaning |
|---|---|
| `TypedDict` | Defines dictionary structure |
| `Annotated` | Adds metadata to a type |
| `Annotated TypedDict` | Dictionary structure + metadata |
| `Literal` | Restricts a value to specific choices |
| `NotRequired` | Makes a TypedDict key optional |
| Pydantic | Runtime validation + parsing |
| JSON Schema | Standard JSON-based schema |

---

# 34. One-Minute Revision

Remember these points:

1. `TypedDict` defines the expected structure of a dictionary.
2. `Annotated` attaches additional metadata to a type.
3. `Annotated TypedDict` combines both.
4. `Annotated` does not automatically perform runtime validation.
5. `TypedDict` is mainly for type hints and structure.
6. `Pydantic` is used when actual runtime validation is required.
7. `Literal` can restrict values to predefined choices.
8. `NotRequired` can make a TypedDict field optional.
9. Annotated fields can be useful when describing LLM structured output.
10. The choice depends on whether you need only structure, metadata, validation, or a standard JSON schema.

---

# 35. Final Mental Model

    TypedDict
    = Structure

    Annotated
    = Metadata

    Annotated TypedDict
    = Structure + Metadata

    Literal
    = Allowed choices

    NotRequired
    = Optional key

    Pydantic
    = Structure + Validation + Parsing

    JSON Schema
    = Standard JSON-based schema

---

# 36. Exam / Interview Definition

> **Annotated TypedDict is a combination of Python's `TypedDict` and `Annotated`, where `TypedDict` defines the expected dictionary structure and `Annotated` attaches additional metadata to individual fields. It is useful for describing structured data, including LLM structured-output schemas, but `Annotated` itself does not provide runtime validation.**

---

# 37. Shortest Possible Revision

    TypedDict
    → Defines dictionary structure.

    Annotated
    → Adds metadata.

    Annotated TypedDict
    → Defines dictionary structure + field metadata.

    Pydantic
    → Validates and parses data at runtime.

    JSON Schema
    → Defines a standard JSON-based schema.

    Most important:
    Annotated metadata ≠ runtime validation.